# WaveletSurfaceNet — the *detail* run: make the learned residual earn its keep

The anchor-only ablation showed the released network is a near-idle residual: trained toward the region-composed
anchor and selected by SDF-error against that same anchor, it just reproduces it. This notebook trains the fork
that could give the network a **genuine** job:

- **Non-anchor, GT-like target.** The residual is trained toward the *dense* field (direct TSDF from a
  `DENSE_N`-point resampling of each mesh), which carries the sub-region relief the **sparse** region-composition
  smears away (the knurl diamonds region-growing can't segment). The identity-start anchor stays the sparse
  region-composed field (sharp edges for free).
- **GT-based selection + value tracking.** Checkpoints are kept by ground-truth reconstruction F-score, and every
  epoch prints **full-model vs anchor-only** on a detail-heavy val set (canonical shapes incl. the **knurl**).

**The run succeeds only if `full > anchor`** — that is the whole point. Saves weights + the value plots + a
knurl full-vs-anchor render to Drive.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch; assert torch.cuda.is_available(), "need a GPU runtime (A100-80GB recommended)"
print("bf16 supported:", torch.cuda.is_bf16_supported())

In [ ]:
REPO_URL = "https://github.com/OlegJakushkin/Points_as_supertoroids.git"   # redirects to WaveletSurfaceNet
BRANCH   = "main"
DRIVE_WS = "/content/drive/MyDrive/wsn_detail"     # persistent workspace in Drive (separate from wsn_composed)
LOCAL    = "/content/wsn"

OUT     = "waveshape_detail"   # checkpoints -> assets/<OUT>.pt (best-by-GT-F) + assets/<OUT>_latest.pt
EPOCHS  = 20
BATCH   = 8                    # A100-80GB, bf16. The DENSE target (tsdf_from_clouds on DENSE_N pts) adds memory
#                                vs the composed run, so keep BATCH ~8-10 (B = 2*BATCH clean+noisy draws).
DENSE_N = 8192                 # dense TARGET cloud size; must exceed the 1536 sparse INPUT so there is genuinely
#                                more detail to recover. 16384 gives more headroom at ~2x the target cost.
print(f"config OK | epochs {EPOCHS} | batch {BATCH} | dense target {DENSE_N} pts vs 1536 sparse input")

In [ ]:
import os
from google.colab import drive
drive.mount("/content/drive")
os.makedirs(DRIVE_WS, exist_ok=True)

In [ ]:
import os, subprocess
if os.path.isdir(LOCAL):
    subprocess.run(["git", "-C", LOCAL, "fetch", "--all"], check=True)
    subprocess.run(["git", "-C", LOCAL, "checkout", BRANCH], check=True)
    subprocess.run(["git", "-C", LOCAL, "pull"], check=True)
else:
    subprocess.run(["git", "clone", "-b", BRANCH, REPO_URL, LOCAL], check=True)
os.chdir(LOCAL); print("repo at", os.getcwd())

In [ ]:
import os, shutil                                    # persist assets/, data/, renders/ on Drive across sessions
for sub in ["assets", "data", "renders"]:
    dst = f"{DRIVE_WS}/{sub}"; os.makedirs(dst, exist_ok=True)
    src = f"{LOCAL}/{sub}"
    if os.path.islink(src): continue
    if os.path.isdir(src):
        for f in os.listdir(src):
            if not os.path.exists(f"{dst}/{f}"): shutil.move(f"{src}/{f}", f"{dst}/{f}")
        shutil.rmtree(src)
    os.symlink(dst, src)
print("assets/, data/, renders/ -> Drive")

In [ ]:
!pip install -q trimesh rtree scikit-image networkx scipy matplotlib bitsandbytes
import trimesh, torch
print("trimesh", trimesh.__version__, "| torch", torch.__version__)
try:
    import bitsandbytes as bnb; print("bitsandbytes", bnb.__version__, "-> Adam8bit ACTIVE")
except Exception as e:
    print("bitsandbytes unavailable (", e, ") -> falls back to fp32-state Adam")

## Build the ALIGNED sparse + dense cloud caches

Both caches are built from the **same mesh list in the same order**, skipping a mesh in *both* if either sampling
fails — so `se_clouds.pt[i]` (1536-pt input/anchor) and `se_clouds_dense.pt[i]` (`DENSE_N`-pt target) are always
the same shape. ModelNet40 raw is downloaded to **local** disk (never Drive); only the two `.pt` caches persist
to Drive. Skips entirely if both caches already exist.

In [ ]:
import os, glob, zipfile, urllib.request
import numpy as np, torch
from waveshape import eval3d as E
from waveshape.datasets import load_mesh_normalized

SPARSE, CAP = 1536, 9843
RAW = "/content/mn40"
need_sparse = not os.path.exists("data/se_clouds.pt")
need_dense  = not os.path.exists("data/se_clouds_dense.pt")

if need_sparse or need_dense:
    if not os.path.isdir(f"{RAW}/ModelNet40"):
        os.makedirs(RAW, exist_ok=True); z = f"{RAW}/ModelNet40.zip"
        if not os.path.exists(z):
            print("downloading ModelNet40 (~2GB) to LOCAL disk...", flush=True)
            urllib.request.urlretrieve("http://modelnet.cs.princeton.edu/ModelNet40.zip", z)
        print("extracting...", flush=True)
        with zipfile.ZipFile(z) as zf: zf.extractall(RAW)
        os.remove(z)
    files = sorted(glob.glob(f"{RAW}/ModelNet40/*/train/*.off"))[:CAP]
    print(f"building ALIGNED caches from {len(files)} meshes (sparse 1536 + dense {DENSE_N})...", flush=True)
    Ps, Ns, Pds, Nds = [], [], [], []
    for i, p in enumerate(files):
        try:
            m = load_mesh_normalized(p, max_faces=200000)
            Pp, Nn = E.sample_cloud(m, n=SPARSE, noise=0.0, seed=0)          # sparse input/anchor
            Pv, Nv = E.sample_cloud(m, n=DENSE_N, noise=0.0, seed=1)         # dense target (diff seed = independent sample)
            if not (np.isfinite(Pp).all() and np.isfinite(Nn).all() and np.isfinite(Pv).all() and np.isfinite(Nv).all()):
                continue                                                      # skip in BOTH -> stays aligned
            Ps.append(Pp.astype(np.float32)); Ns.append(Nn.astype(np.float32))
            Pds.append(Pv.astype(np.float32)); Nds.append(Nv.astype(np.float32))
        except Exception:
            continue
        if (i + 1) % 1000 == 0: print(f"  {i+1}/{len(files)} ({len(Ps)} ok)", flush=True)
    torch.save({"P": torch.tensor(np.stack(Ps)),  "N": torch.tensor(np.stack(Ns))},  "data/se_clouds.pt")
    torch.save({"P": torch.tensor(np.stack(Pds)), "N": torch.tensor(np.stack(Nds))}, "data/se_clouds_dense.pt")
    print(f"cached {len(Ps)} aligned sparse+dense clouds to Drive", flush=True)
else:
    print("both caches already on Drive -- skipping download", flush=True)
# sanity: alignment + densities
s = torch.load("data/se_clouds.pt", weights_only=False); d = torch.load("data/se_clouds_dense.pt", weights_only=False)
assert s["P"].shape[0] == d["P"].shape[0], "MISALIGNED caches -- delete both and rebuild"
print(f"sparse {tuple(s['P'].shape)}  |  dense {tuple(d['P'].shape)}  (aligned: {s['P'].shape[0]==d['P'].shape[0]})")

## Train — residual toward the dense GT-like target, GT-based selection

`train_detail.py` builds the sparse region-composed anchor (identity start) and trains the residual toward the
dense field. Auto-resumes across Colab sessions from `assets/<OUT>_latest.pt`. Watch the per-epoch line:
**`GT-F full X vs ANCHOR Y (d±Δ)`** and the **KNURL** breakdown — a positive Δ is the network earning its keep.

In [ ]:
import os
os.environ["MPLBACKEND"] = "Agg"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
latest = f"assets/{OUT}_latest.pt"
args = f"--out {OUT} --batch {BATCH} --epochs {EPOCHS} --dense data/se_clouds_dense.pt --target dense"
if os.path.exists(latest):
    args += f" --resume {latest}"
print("python train_detail.py", args, flush=True)
!python train_detail.py {args} 2>&1 | tee -a {DRIVE_WS}/train_{OUT}.log

## Did the network earn its keep? — the value plot

Full-model vs anchor-only ground-truth F-score across epochs (left), and the per-shape gap at the best epoch
(right). The knurl bar is the headline: if the residual recovers relief the composition can't, its full bar
clears its anchor bar.

In [ ]:
import json, numpy as np
import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
H = json.load(open("renders/detail_train_hist.json"))
ep = [h["epoch"] for h in H]; full = [h["val_f_full"] for h in H]; anch = [h["val_f_anchor"] for h in H]
best = max(range(len(H)), key=lambda i: H[i]["val_f_full"]); ps = H[best]["per_shape"]
fig, (a1, a2) = plt.subplots(1, 2, figsize=(13, 4.2))
a1.plot(ep, full, "-o", color="#c0392b", label="full model", lw=2)
a1.plot(ep, anch, "--o", color="#e67e22", label="anchor-only (network OFF)", lw=2)
a1.axvline(H[best]["epoch"], color="#999", ls=":", lw=1)
a1.set_xlabel("epoch"); a1.set_ylabel(f"ground-truth F-score"); a1.set_title("Does the residual add value?"); a1.legend(); a1.grid(alpha=.3)
names = list(ps.keys()); fv = [ps[n][0] for n in names]; av = [ps[n][1] for n in names]
x = np.arange(len(names)); w = 0.4
a2.bar(x-w/2, av, w, color="#e67e22", label="anchor-only")
a2.bar(x+w/2, fv, w, color="#c0392b", label="full")
a2.set_xticks(x); a2.set_xticklabels(names, rotation=45, ha="right"); a2.set_ylabel("GT F-score")
a2.set_title(f"Per-shape at best epoch {H[best]['epoch']}"); a2.legend(); a2.grid(alpha=.3, axis="y")
fig.tight_layout(); fig.savefig("renders/detail_value.png", dpi=140, bbox_inches="tight")
knf, kna = ps.get("knurl", [0,0])
verdict = "EARNS ITS KEEP" if H[best]["val_f_full"] > H[best]["val_f_anchor"] else "does NOT yet beat anchor"
print(f"best epoch {H[best]['epoch']}: full {H[best]['val_f_full']:.2f} vs anchor {H[best]['val_f_anchor']:.2f} "
      f"(d {H[best]['val_f_full']-H[best]['val_f_anchor']:+.2f})  ->  network {verdict}")
print(f"KNURL at best: full {knf} vs anchor {kna} (d {knf-kna:+.1f})")
plt.show()

## Knurl, up close — anchor vs the trained residual (same regions, GT overlay)

In [ ]:
import numpy as np, torch, trimesh
import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d.art3d import Poly3DCollection
from waveshape import wavelet as WV, eval3d as E, shapes as S
DEV, BOUND, TRUNC, RES = "cuda", 1.1, 0.1, 128
m = S.normalize_to_unit_cube(E._knurl_mesh())
P, N = E.sample_cloud(m, n=1536, noise=0.0, seed=0)
Pt = torch.tensor(P[None]).float().to(DEV); Nt = torch.tensor(N[None]).float().to(DEV)
net = WV.load_at_res(torch.load(f"assets/{OUT}.pt", weights_only=False), res=RES, bound=BOUND).to(DEV).eval()
with torch.no_grad():
    out, c_anchor, _, _ = net(Pt, Nt)
    araw = net._postprocess(WV.idwt3d(c_anchor, net.haar), Pt)[0,0].float().cpu().numpy()*TRUNC
    traw = out[0,0].float().cpu().numpy()*TRUNC
va, fa = WV.mesh_field(araw, "mixed", bound=BOUND, trunc=TRUNC)
vt, ft = WV.mesh_field(traw, "mixed", bound=BOUND, trunc=TRUNC)
_L = np.array([0.4,-0.6,0.72]); _L/=np.linalg.norm(_L)
def draw(ax, v, f, c, title):
    tri = v[f]; fn = np.cross(tri[:,1]-tri[:,0], tri[:,2]-tri[:,0]); fn/=np.clip(np.linalg.norm(fn,axis=1,keepdims=True),1e-9,None)
    sh = np.clip(np.abs(fn@_L)*0.5+0.5,0.3,1.0)
    ax.add_collection3d(Poly3DCollection(tri, facecolors=np.clip(c[None]*sh[:,None],0,1), edgecolor="none"))
    ax.set_xlim(-1,1);ax.set_ylim(-1,1);ax.set_zlim(-1,1);ax.set_axis_off();ax.set_box_aspect((1,1,1));ax.view_init(18,-60);ax.set_title(title,fontsize=12)
fig = plt.figure(figsize=(12,4.5))
draw(fig.add_subplot(1,3,1,projection="3d"), m.vertices/np.abs(m.vertices).max(), m.faces, np.array([0.5,0.55,0.6]), "ground truth")
draw(fig.add_subplot(1,3,2,projection="3d"), va, fa, np.array([0.62,0.42,0.42]), "anchor-only (network OFF)")
draw(fig.add_subplot(1,3,3,projection="3d"), vt, ft, np.array([0.40,0.52,0.62]), "full trained residual")
fig.suptitle("Knurl relief: can the residual recover what region-growing can't segment?", y=1.02, fontsize=13)
fig.tight_layout(); fig.savefig("renders/detail_knurl.png", dpi=140, bbox_inches="tight"); plt.show()

## Zip weights + value plots + history to Drive

In [ ]:
import glob, os, zipfile
zpath = f"{DRIVE_WS}/wsn_detail_result.zip"
with zipfile.ZipFile(zpath, "w", zipfile.ZIP_DEFLATED) as z:
    for pat in [f"assets/{OUT}.pt", f"assets/{OUT}_latest.pt", "renders/detail_train_hist.json",
                "renders/detail_value.png", "renders/detail_knurl.png"]:
        for f in glob.glob(pat):
            if os.path.exists(f): z.write(f, arcname=os.path.basename(f)); print("  +", f)
print("saved:", zpath, f"({os.path.getsize(zpath)/1e6:.1f} MB)")